# Train a Model from labeled clips (starter)

TODO(template): a starting point for your own Model's training. Experiment here, then
export to ONNX and point `[models.<name>] file` at the result.

It reads the labels from `make label` (`labels/<model>.csv` in the private data folder),
takes a few frames from each clip, trains a small Keras image classifier and exports it
to ONNX. Run it with `make notebook` (needs the `train` extra).

Model X itself is a ready-trained detector (`make model-x`); use this notebook when you
make Model Y or replace Model X with your own.

In [ ]:
from pathlib import Path

import numpy as np

from camwatch.config import load_config
from camwatch.labeling.clip_labels import ClipLabels
from camwatch.video import read_frames

config = load_config(Path("../../../../config.toml"))  # from src/camwatch/models/model_x/
MODEL = config.labeling.model
clips = config.paths.private(config.labeling.clips)
labels = ClipLabels(clips, config.paths.private(Path("labels") / f"{MODEL}.csv")).labeled()
names = sorted(set(labels.values()))
print(f"{len(labels)} labeled clips for {MODEL}: {names}")

## Frames and labels

In [ ]:
SIZE = 128  # pixels square
EVERY_SECONDS = 1.0  # one frame per second of each clip


def small(image: np.ndarray) -> np.ndarray:
    """`image` resized to SIZE x SIZE."""
    rows = np.arange(SIZE) * image.shape[0] // SIZE
    cols = np.arange(SIZE) * image.shape[1] // SIZE
    return image[rows][:, cols]


pictures, targets, groups = [], [], []
for clip, label in labels.items():
    for _, image in read_frames(clips / clip, EVERY_SECONDS):
        pictures.append(small(image))
        targets.append(names.index(label))
        groups.append(clip)
x = np.stack(pictures).astype(np.float32) / 255
y = np.array(targets)
print(x.shape, np.bincount(y))

## Hold out whole clips to measure accuracy

In [ ]:
rng = np.random.default_rng(0)
held_out = set(rng.choice(sorted(set(groups)), size=max(1, len(set(groups)) // 5), replace=False))
test = np.array([g in held_out for g in groups])
x_train, y_train, x_test, y_test = x[~test], y[~test], x[test], y[test]
print(len(x_train), "to train,", len(x_test), "to test")

## Train

In [ ]:
import tensorflow as tf

model = tf.keras.Sequential(
    [
        tf.keras.Input((SIZE, SIZE, 3)),
        tf.keras.layers.Conv2D(16, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(len(names), activation="softmax"),
    ]
)
model.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(x_train, y_train, epochs=10, validation_data=(x_test, y_test))

## Export to ONNX

In [ ]:
import tf2onnx

out = config.paths.private(Path("models") / f"{MODEL}_trained.onnx")
out.parent.mkdir(parents=True, exist_ok=True)
signature = [tf.TensorSpec((None, SIZE, SIZE, 3), tf.float32, name="image")]
tf2onnx.convert.from_keras(model, input_signature=signature, output_path=str(out))
print(f"Wrote {out}; labels in order: {names}")